In [1]:
import os

raw_path = "data/raw"
os.makedirs(raw_path, exist_ok=True)

In [2]:
import pandas as pd

In [3]:
csv = f"{raw_path}/customer_journey.csv"
raw_df = pd.read_csv(csv)
raw_df.head(5)

,SessionID,UserID,Timestamp,PageType,DeviceType,Country,ReferralSource,TimeOnPage_seconds,ItemsInCart,Purchased
0,session_0,user_2223,2025-01-20 22:53:34,home,Desktop,India,Social Media,55,0,0
1,session_1,user_2192,2025-02-26 12:57:10,home,Tablet,Germany,Email,99,0,0
2,session_1,user_2192,2025-02-26 12:59:11,product_page,Tablet,Germany,Email,121,0,0
3,session_2,user_1708,2025-06-24 15:40:46,home,Mobile,India,Google,160,0,0
4,session_3,user_2976,2025-06-11 07:21:02,home,Tablet,UK,Google,113,0,0


In [4]:
raw_df.dtypes

,0
SessionID,object
UserID,object
Timestamp,object
PageType,object
DeviceType,object
Country,object
ReferralSource,object
TimeOnPage_seconds,int64
ItemsInCart,int64
Purchased,int64


In [5]:
raw_df.shape

(12719, 10)

In [6]:
raw_df["PageType"].unique()

array(['home', 'product_page', 'cart', 'checkout', 'confirmation'],
      dtype=object)

In [7]:
raw_df.groupby("PageType")["Purchased"].mean()

,Purchased
PageType,
cart,0.631645
checkout,0.899377
confirmation,1.000000
home,0.202000
product_page,0.253323


In [8]:
def normalize_timestamp(time_column: str, df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  df[time_column] = pd.to_datetime(df[time_column])
  return df

In [9]:
def sort_data(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
  df = df.copy()
  df = df.sort_values(by=columns).reset_index(drop=True)
  return df

In [10]:
funnel_order = {'home': 1, 'product_page': 2, 'cart': 3,
                'checkout': 4, 'confirmation': 5}

def create_funnel_order(df: pd.DataFrame, funnel_column: str) -> pd.DataFrame:
  df = df.copy()
  df["funnel_order"] = df[funnel_column].map(funnel_order)
  return df

In [11]:
silver_df = raw_df.copy()

In [12]:
silver_df = normalize_timestamp("Timestamp", silver_df)
silver_df = sort_data(silver_df, ["SessionID", "Timestamp"])
silver_df = create_funnel_order(silver_df, "PageType")

In [13]:
silver_df.head(5)

,SessionID,UserID,Timestamp,PageType,DeviceType,Country,ReferralSource,TimeOnPage_seconds,ItemsInCart,Purchased,funnel_order
0,session_0,user_2223,2025-01-20 22:53:34,home,Desktop,India,Social Media,55,0,0,1
1,session_1,user_2192,2025-02-26 12:57:10,home,Tablet,Germany,Email,99,0,0,1
2,session_1,user_2192,2025-02-26 12:59:11,product_page,Tablet,Germany,Email,121,0,0,2
3,session_10,user_2357,2025-05-17 22:11:37,home,Tablet,India,Direct,75,0,0,1
4,session_10,user_2357,2025-05-17 22:13:33,product_page,Tablet,India,Direct,116,0,0,2


In [14]:
os.makedirs("data/silver", exist_ok=True)

In [15]:
def save_file(df: pd.DataFrame):
  df.to_csv("data/silver/customer_journey_silver.csv", index=False)

In [16]:
save_file(silver_df)